In [2]:

import json
from collections import defaultdict
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm

# Path to the chunks file
chunks_path = '../data/processed/polish_dataset_7_no_chain/merged/chunks.jsonl'

# Load chunks and group by doc_id
doc_chunks = defaultdict(list)
with open(chunks_path, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line)
        chunk_id = entry['chunk_id']
        context_ids = entry['explicit_context_chunks']
        chunk_text = entry['chunk']
        doc_id = chunk_id.split('_')[0]
        doc_chunks[doc_id].append((chunk_id, context_ids, chunk_text))

print(f"Loaded {len(doc_chunks)} documents.")
for doc_id, chunks in doc_chunks.items():
    print(f"{doc_id}: {len(chunks)} chunks")


/home/mpj/repositories/long-context-retrieval/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 7 documents.
education: 2452 chunks
financing: 1015 chunks
highereducation: 3644 chunks
classified: 795 chunks
military: 2305 chunks
police: 2919 chunks
healthcare: 4344 chunks


In [3]:
import tiktoken

# compute the amount of tokens per dataset
tokenizer = tiktoken.get_encoding("o200k_harmony")
doc_token_counts = {}
doc_word_counts = {}
for doc_id, chunks in doc_chunks.items():
    full_text = " ".join(chunk[-1] for chunk in chunks)
    tokens = tokenizer.encode(full_text)
    doc_token_counts[doc_id] = len(tokens)
    doc_word_counts[doc_id] = len(full_text.split())
# Print token counts
for doc_id, token_count in doc_token_counts.items():
    print(f"{doc_id}: {token_count} tokens")

for doc_id, word_count in doc_word_counts.items():
    print(f"{doc_id}: {word_count} words")
    

education: 164463 tokens
financing: 100774 tokens
highereducation: 172481 tokens
classified: 41730 tokens
military: 136780 tokens
police: 184005 tokens
healthcare: 251843 tokens
education: 73540 words
financing: 45494 words
highereducation: 76970 words
classified: 18035 words
military: 57729 words
police: 79259 words
healthcare: 110980 words


In [22]:
# analyse if there are any chunks with explicit or implicit context that references to a chunk AFTER the position of the chunk
# compute the percentage for which it happens

from sympy.physics.units import l
for doc_id, chunks in doc_chunks.items():
    total = len([chunk for chunk in chunks if chunk[1]])  # only consider chunks with explicit context
    first_occurance = None
    last_occurance = None
    last_occurence_chunk = None
    count_ref_forward = 0
    index = 0

    for chunk_id, context_ids, chunk_text in chunks:
        chunk_idx = int(chunk_id.split('_')[-1])
        context_idxs = [int(cid.split('_')[-1]) for cid in context_ids]
        # context_idxs.append(chunk_idx)  # include the chunk itself as context
        if not context_idxs:
            continue  # skip if no explicit context
        if all(abs(ctx_idx-chunk_idx) < 300 for ctx_idx in context_idxs):
            count_ref_forward += 1
            if first_occurance is None:
                first_occurance = index / total
                # print(chunk_id, context_ids)
                # print(chunk_idx, context_idxs)
            last_occurance = index / total
            last_occurence_chunk = chunk_id
        index += 1
    percentage = (count_ref_forward / total) * 100 if total > 0 else 0
    print(f"{doc_id}: {percentage:.2f}% of chunks reference a chunk after their position")
    print(f"First occurence: {first_occurance:.3f} (percentile)")
    print(f"Last occurence: {last_occurance:.2f} (percentile) for chunk {last_occurence_chunk}")

        
    

education: 77.41% of chunks reference a chunk after their position
First occurence: 0.001 (percentile)
Last occurence: 1.00 (percentile) for chunk education_law_2449
financing: 85.68% of chunks reference a chunk after their position
First occurence: 0.000 (percentile)
Last occurence: 0.99 (percentile) for chunk financing_education_1010
highereducation: 70.73% of chunks reference a chunk after their position
First occurence: 0.000 (percentile)
Last occurence: 1.00 (percentile) for chunk highereducation_science_3642
classified: 90.10% of chunks reference a chunk after their position
First occurence: 0.000 (percentile)
Last occurence: 0.99 (percentile) for chunk classified_information_786
military: 88.34% of chunks reference a chunk after their position
First occurence: 0.000 (percentile)
Last occurence: 1.00 (percentile) for chunk military_2292
police: 86.56% of chunks reference a chunk after their position
First occurence: 0.000 (percentile)
Last occurence: 1.00 (percentile) for chunk p